# Das evidências às oportunidades de melhoria

**Técnico em Ciência de Dados — 2º ano**  
Análise Exploratória de Dados e Inteligência de Negócios  
U3 — Projeto Análise Exploratória de Dados  
S13 — Projeto AED – Desenvolvimento das Análises  
Aula 3: Interpretação dos resultados da investigação

**Objetivo:** Explicar os resultados da investigação considerando a pergunta e as evidências obtidas no projeto.

Vocês integram a equipe de análise de dados de uma rede brasileira de oito papelarias. A direção recebeu as descobertas sobre atendimento, disponibilidade de produtos e distribuição dos estoques. Agora precisa compreender o significado dessas evidências para o desempenho comercial e para o planejamento das operações futuras.

**Pergunta investigativa:** Quais condições estão associadas ao desempenho comercial das lojas e qual aspecto merece prioridade para uma primeira ação de melhoria?

**Missão:** Expliquem os resultados das investigações e identifiquem como a rede poderia atender melhor à procura e utilizar seus estoques de forma mais eficiente.

Dados simulados para fins didáticos. Período: 06/10 a 30/11/2025. Tempo: 43 minutos.

## Organização do ambiente e materiais

Organizem-se em duplas, com um computador por dupla. Alternem execução e interpretação. Reúnam este notebook e os dois CSVs na mesma pasta; selecionem o kernel Python no VS Code/Jupyter e executem as células na ordem. Dependências: pandas, numpy e IPython (ambiente Jupyter). Não são necessários arquivos produzidos em outras aulas.

Arquivos: `rede_papelarias_B2_integrado.csv`, `rede_papelarias_produtos_S13A2.csv`.

## 1. Recuperem as evidências — 5 min

O código recupera os registros, elimina a duplicata exata, normaliza valores monetários, agrega os seis produtos por loja-dia e recompõe os indicadores. Aproveitamento = cupons por 100 entradas; resultado após despesas = lucro bruto menos divulgação, pessoal e outras despesas operacionais registradas. Não equivale necessariamente ao lucro líquido. As ausências não são preenchidas.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

arquivos = [Path("rede_papelarias_B2_integrado.csv"), Path("rede_papelarias_produtos_S13A2.csv")]
for arquivo in arquivos:
    if not arquivo.is_file():
        raise FileNotFoundError(f"Coloquem {arquivo.name} na mesma pasta do notebook.")
original = pd.read_csv(arquivos[0], encoding="utf-8-sig")
produtos = pd.read_csv(arquivos[1], encoding="utf-8-sig", parse_dates=["data"])
# Recompor a preparação da A1, preservando ausências.
lojas = original.drop_duplicates().copy()
assert lojas.id_registro.is_unique
for coluna in ["despesas_com_pessoal", "outras_despesas_operacionais"]:
    antes = lojas[coluna]
    texto = antes.astype("string").str.strip().str.replace("R$", "", regex=False).str.strip().str.replace(",", ".", regex=False)
    convertido = pd.to_numeric(texto, errors="raise")
    assert convertido.isna().sum() == antes.isna().sum()
    lojas[coluna] = convertido.astype(float)
lojas["data"] = pd.to_datetime(lojas.data)
lojas = lojas.loc[lojas.data.between("2025-10-06", "2025-11-30")].copy()
assert lojas.entradas.gt(0).all()
lojas["cupons_por_100_entradas"] = 100 * lojas.cupons_emitidos / lojas.entradas
lojas["resultado_apos_despesas"] = (lojas.lucro_bruto - lojas.investimento_divulgacao
    - lojas.despesas_com_pessoal - lojas.outras_despesas_operacionais)

# Conferir chaves, saldos e vínculo com cada loja-dia.
produtos = produtos.sort_values(["loja", "produto", "data"]).copy()
assert produtos.id_registro_produto.is_unique
assert not produtos.duplicated(["loja", "produto", "data"]).any()
assert not produtos.isna().any().any()
assert produtos.groupby(["loja", "data"]).size().eq(6).all()
campos_un = ["estoque_inicial_un", "reposicao_un", "unidades_vendidas", "unidades_nao_atendidas", "estoque_final_un"]
assert produtos[campos_un].ge(0).all().all()
assert all(pd.api.types.is_integer_dtype(produtos[c]) for c in campos_un)
assert (produtos.estoque_inicial_un + produtos.reposicao_un - produtos.unidades_vendidas == produtos.estoque_final_un).all()
anterior = produtos.groupby(["loja", "produto"]).estoque_final_un.shift()
assert produtos.loc[anterior.notna(), "estoque_inicial_un"].eq(anterior.dropna()).all()
assert produtos.ruptura_no_dia.eq((produtos.estoque_inicial_un.eq(0) | produtos.estoque_final_un.eq(0)).astype(int)).all()
assert (produtos.unidades_nao_atendidas.eq(0) | produtos.estoque_final_un.eq(0)).all()
vinculos = produtos.merge(lojas[["id_registro", "loja", "data"]], left_on=["id_registro_loja_dia", "loja", "data"], right_on=["id_registro", "loja", "data"], how="left", validate="many_to_one")
assert vinculos.id_registro.notna().all()
# Agregar os produtos ANTES de juntar aos indicadores comerciais.
produtos["procura"] = produtos.unidades_vendidas + produtos.unidades_nao_atendidas
produtos["semana"] = (produtos.data.sub(pd.Timestamp("2025-10-06")).dt.days // 7 + 1).astype(int)
produtos["cmv_sel"] = produtos.unidades_vendidas * produtos.custo_unitario_aquisicao
produtos["abertura_zero"] = produtos.estoque_inicial_un.eq(0).astype(int)
diario = produtos.groupby(["loja", "data"], as_index=False).agg(vendidas=("unidades_vendidas", "sum"), nao_atendidas=("unidades_nao_atendidas", "sum"), procura=("procura", "sum"), receita_sel=("receita_liquida_produto", "sum"), cmv_sel=("cmv_sel", "sum"), abertura_zero=("abertura_zero", "sum"))
integrado = lojas.merge(diario, on=["loja", "data"], how="left", validate="one_to_one")
assert len(integrado) == len(lojas) == len(diario)
assert integrado.procura.notna().all()
assert np.isclose(integrado.receita_liquida.sum(), lojas.receita_liquida.sum())
assert integrado.resultado_apos_despesas.isna().sum() == lojas.resultado_apos_despesas.isna().sum()
assert (integrado.receita_sel <= integrado.receita_liquida + .001).all()
assert (integrado.cmv_sel <= integrado.receita_liquida - integrado.lucro_bruto + .001).all()
assert (integrado.receita_sel - integrado.cmv_sel <= integrado.lucro_bruto + .001).all()
assert (integrado.receita_sel >= integrado.cmv_sel - .001).all()
assert integrado.loc[integrado.disponibilidade_produtos_pct.eq(100), "abertura_zero"].eq(0).all()

print("Loja-dias:",len(integrado),"| Registros de produtos:",len(produtos))
# Proporção pela razão entre totais; denominador zero permanece indefinido.
chave = ["loja", "produto"]
por_produto = produtos.groupby(chave).agg(descricao=("descricao_produto", "first"), procura=("procura", "sum"), vendas=("unidades_vendidas", "sum"), nao_atendidas=("unidades_nao_atendidas", "sum"), estoque_mediano=("estoque_final_un", "median"))
por_produto["pct_nao_atendida"] = 100 * por_produto.nao_atendidas / por_produto.procura.replace(0, np.nan)
semanal = produtos.groupby(chave + ["semana"]).agg(procura=("procura", "sum"), nao_atendidas=("unidades_nao_atendidas", "sum"), estoque_fim=("estoque_final_un", "last"))
por_produto["semanas_com_falta"] = semanal.nao_atendidas.gt(0).groupby(level=[0,1]).sum()
ordem = por_produto.sort_values("nao_atendidas", ascending=False)

# Triagem exploratória: referência de duas semanas de vendas médias.
por_produto["referencia_un"] = np.ceil(2 * por_produto.vendas / 8).astype(int)
semanal_ref = semanal.join(por_produto.referencia_un)
por_produto["semanas_acima"] = semanal_ref.estoque_fim.gt(semanal_ref.referencia_un).groupby(level=[0,1]).sum()
por_produto["candidato"] = por_produto.estoque_mediano.gt(por_produto.referencia_un) & por_produto.semanas_acima.ge(6)
candidatos = por_produto.loc[por_produto.candidato].copy()
rotulos = {"descricao":"Produto", "vendas":"Vendas em 8 semanas (un.)", "referencia_un":"Referência (un.)", "estoque_mediano":"Estoque final mediano (un.)", "semanas_acima":"Fechamentos acima da referência (de 8)"}

# Procurar faltas e estoques do mesmo produto em outras lojas na mesma data.
com_ref = produtos.join(por_produto[["referencia_un", "candidato"]], on=["loja", "produto"])
falta = com_ref.loc[com_ref.unidades_nao_atendidas.gt(0), ["data", "produto", "loja", "unidades_nao_atendidas"]].rename(columns={"loja":"loja_falta"})
estoque = com_ref.loc[com_ref.candidato & com_ref.estoque_inicial_un.gt(com_ref.referencia_un) & com_ref.estoque_final_un.gt(com_ref.referencia_un) & com_ref.ruptura_no_dia.eq(0), ["data", "produto", "loja"]].rename(columns={"loja":"loja_estoque"})
coincidencias = falta.merge(estoque, on=["data", "produto"]).query("loja_falta != loja_estoque")
pares = coincidencias.groupby(["produto", "loja_falta", "loja_estoque"]).agg(dias_coincidentes=("data", "nunique"), unidades_nas_datas=("unidades_nao_atendidas", "sum")).sort_values("dias_coincidentes", ascending=False)

print("Contrastes de estoque e procura não atendida — oito semanas")
display(pares.head(2).rename(columns={"dias_coincidentes":"Dias coincidentes", "unidades_nas_datas":"Unidades não atendidas"}))
print("Critério de triagem: mediana acima de duas semanas de vendas médias e pelo menos seis fechamentos semanais acima da referência; nas datas coincidentes, saldo de abertura e fechamento acima da referência, sem ruptura.")


### Dicionário dos dados complementares

| Campo | Unidade/tipo | Definição |
|---|---|---|
| id_registro_produto | texto | Identificador único do registro de produto. |
| id_registro_loja_dia | texto | Identificador do registro da loja e dia no CSV integrado da A1. |
| data | AAAA-MM-DD | Data da operação da loja. |
| loja | texto | Identificador da loja, L1 a L8. |
| produto | texto | Identificador do produto, P01 a P06. |
| descricao_produto | texto | Denominação comercial simulada. |
| estoque_inicial_un | inteiro, unidades | Estoque na abertura, antes de reposições do dia. |
| reposicao_un | inteiro, unidades | Quantidade recebida após a abertura e antes das vendas nesse modelo. Não inclui transferências entre lojas. |
| unidades_vendidas | inteiro, unidades | Unidades do produto vendidas no dia. Não equivale a número de cupons. |
| unidades_nao_atendidas | inteiro, unidades | Quantidade solicitada e não fornecida por indisponibilidade do produto (registro de solicitações, não clientes distintos nem vendas perdidas). |
| ruptura_no_dia | inteiro (0/1) | 1 se o estoque esteve zerado na abertura ou atingiu zero após as vendas; 0 caso contrário. Pode haver ruptura sem solicitação registrada não atendida. |
| estoque_final_un | inteiro, unidades | Saldo no fechamento do dia. |
| custo_unitario_aquisicao | R$/unidade | Custo simulado de aquisição unitária, constante por produto. |
| receita_liquida_produto | R$ | Receita do produto no dia, descontada a campanha simulada. |
| volume_unitario_l | litros/unidade | Volume externo aproximado por unidade embalada, assumido na simulação; não inclui corredores, prateleiras ou folgas. |

A disponibilidade geral refere-se à abertura da loja. Os seis produtos não representam todo o sortimento nem toda a procura. O resultado após despesas registradas não equivale a lucro líquido; cupons por 100 entradas não identifica pessoas distintas.

## 2. Expliquem os resultados integrados — 12 min

Comparem o período completo com a faixa de 150 a 199 entradas, usando os mesmos casos completos para as quatro condições. As comparações de lojas não são pareadas por data. A proporção não atendida considera os seis produtos e é calculada pela razão dos totais.

In [ ]:
# Preservar o conjunto de casos completos da A1 e conferir o denominador adicional.
integrado["pct_nao_atendida"] = 100 * integrado.nao_atendidas / integrado.procura.replace(0, np.nan)
condicoes = ["espera_media_min", "horas_equipe", "disponibilidade_produtos_pct", "pct_nao_atendida"]
desempenho = ["cupons_por_100_entradas", "receita_liquida", "resultado_apos_despesas"]
variaveis_a1 = ["espera_media_min", "horas_equipe", "disponibilidade_produtos_pct"] + desempenho
completos_a1 = integrado.dropna(subset=variaveis_a1).copy()
completos = completos_a1.dropna(subset=["pct_nao_atendida"]).copy()
faixa = completos.loc[completos.entradas.between(150, 199)].copy()
print("Loja-dias no período:", len(integrado))
print("Casos completos da A1:", len(completos_a1))
print("Casos completos comuns:", len(completos))
print("Casos completos comuns na faixa de 150 a 199 entradas:", len(faixa))
rotulos_corr = {"espera_media_min":"Espera média", "horas_equipe":"Horas de equipe", "disponibilidade_produtos_pct":"Disponibilidade geral", "pct_nao_atendida":"Procura não atendida (%)", "cupons_por_100_entradas":"Cupons por 100 entradas", "receita_liquida":"Receita líquida", "resultado_apos_despesas":"Resultado após despesas"}
def correlacoes(tab):
    return tab[condicoes + desempenho].corr(method="pearson").loc[condicoes, desempenho]
corr_periodo, corr_faixa = correlacoes(completos), correlacoes(faixa)
print("Pearson — período completo")
display(corr_periodo.rename(index=rotulos_corr, columns=rotulos_corr).round(3))
print("Pearson — faixa de movimento")
display(corr_faixa.rename(index=rotulos_corr, columns=rotulos_corr).round(3))
# Comparar as oito lojas; proporção agregada pela razão dos totais.
def resumo_lojas(tab):
    res = tab.groupby("loja").agg(n=("id_registro", "size"), disponibilidade=("disponibilidade_produtos_pct", "mean"), espera=("espera_media_min", "mean"), aproveitamento=("cupons_por_100_entradas", "mean"), receita=("receita_liquida", "mean"), resultado=("resultado_apos_despesas", "mean"), procura=("procura", "sum"), nao_atendidas=("nao_atendidas", "sum"))
    res["pct_nao_atendida"] = 100 * res.nao_atendidas / res.procura.replace(0, np.nan)
    return res
lojas_periodo, lojas_faixa = resumo_lojas(completos), resumo_lojas(faixa)
rotulos = {"n":"Observações", "disponibilidade":"Disponibilidade geral (%)", "espera":"Espera média (min)", "nao_atendidas":"Não atendida (un.)", "procura":"Procura (un.)", "pct_nao_atendida":"Não atendida (%)", "aproveitamento":"Cupons por 100 entradas", "receita":"Receita média (R$)", "resultado":"Resultado médio (R$)"}
print("Quatro lojas — faixa de 150 a 199 entradas")
display(lojas_faixa.loc[["L3","L4","L5","L8"], ["n","disponibilidade","pct_nao_atendida","aproveitamento","receita","resultado"]].rename(columns=rotulos).round(2))

**Q1.** Expliquem por que as condições associadas ao aproveitamento ou à receita não apresentam necessariamente a mesma relação com o resultado após despesas. Usem as correlações de espera, horas de equipe e disponibilidade e comparem L3 com L4 e L5 com L8.

**Resposta:**



## 3. Comparem as alternativas de abastecimento — 13 min

Na última semana observada (24 a 29/11/2025), comparem duas formas hipotéticas de atender à mesma procura adicional: comprar mais produtos ou aproveitar os estoques da rede. O código mantém as vendas e reposições registradas, verifica a viabilidade dos saldos diários e pressupõe disponibilidade no momento necessário. O cenário não é uma ordem de transferência atual.

In [ ]:
# Cenário retrospectivo: mesmos dias, vendas e reposições observados.
# O aproveitamento de estoque é representado por uma alocação no início da semana.
semana_final = produtos.loc[produtos.data.between("2025-11-24", "2025-11-29")].copy()
casos = [("P01", "L3", "L4"), ("P03", "L5", "L2")]
linhas = []
for cod, destino, origem in casos:
    d = semana_final.query("produto == @cod and loja == @destino").sort_values("data")
    o = semana_final.query("produto == @cod and loja == @origem").sort_values("data")
    h = produtos.query("produto == @cod and loja == @destino")
    assert len(d) == len(o) == 6
    n = int(d.unidades_nao_atendidas.sum())
    custo = h.custo_unitario_aquisicao.unique(); volume = h.volume_unitario_l.unique()
    assert len(custo) == len(volume) == 1
    custo, volume = float(custo[0]), float(volume[0])
    assert o.custo_unitario_aquisicao.eq(custo).all() and o.volume_unitario_l.eq(volume).all()
    preco = h.receita_liquida_produto.sum() / h.unidades_vendidas.sum()
    # Saldo de origem após destinar n unidades; saldo de destino após atender a procura.
    origem_abertura = o.estoque_inicial_un - n
    origem_fechamento = o.estoque_final_un - n
    destino_fechamento = d.estoque_final_un + n - d.unidades_nao_atendidas.cumsum()
    destino_abertura = d.estoque_inicial_un + n - d.unidades_nao_atendidas.cumsum().shift(fill_value=0)
    assert min(origem_abertura.min(), origem_fechamento.min()) > 0
    assert min(destino_abertura.min(), destino_fechamento.min()) >= 0
    saldo_compra = int(o.estoque_final_un.iloc[-1] + destino_fechamento.iloc[-1])
    saldo_aproveita = int(origem_fechamento.iloc[-1] + destino_fechamento.iloc[-1])
    linhas.append({"Produto": h.descricao_produto.iloc[0], "Loja com falta": destino, "Loja com estoque": origem,
        "Unidades adicionais": n, "Compras adicionais (R$)": n*custo,
        "Estoque final — compra (un.)": saldo_compra, "Estoque final — aproveitamento (un.)": saldo_aproveita,
        "Receita adicional (R$)": n*preco, "Lucro bruto adicional (R$)": n*(preco-custo), "Volume embalado a menos (L)": n*volume})
cenario = pd.DataFrame(linhas).set_index("Produto")
print("Procura registrada não atendida — 24 a 29/11/2025")
display(cenario[["Loja com falta", "Loja com estoque", "Unidades adicionais"]])
print("Comparação das alternativas (mesmo atendimento adicional)")
comparacao = pd.DataFrame({"Compra adicional": [cenario["Unidades adicionais"].sum(), cenario["Compras adicionais (R$)"].sum(), cenario["Estoque final — compra (un.)"].sum()],
    "Aproveitamento de estoques existentes": [0, 0, cenario["Estoque final — aproveitamento (un.)"].sum()]}, index=["Compras adicionais (un.)", "Desembolso em compras adicionais (R$)", "Estoque final (un.)"])
display(comparacao.round(2))


**Q2.** Na semana examinada, comparem as duas formas de atender às 32 unidades de procura não atendida. Qual delas exige menos compras adicionais? Quanto a rede deixaria de desembolsar em compras e qual seria a quantidade final de mercadorias em estoque em cada alternativa?

**Resposta:**



## 4. Interpretem as oportunidades econômicas — 6 min

Lucro bruto adicional = receita adicional − custo de aquisição das unidades vendidas. O preço líquido médio é a receita registrada dividida pelas unidades vendidas nas oito semanas, para cada produto e loja de destino. A diferença de volume corresponde às embalagens; o desembolso evitado não é lucro. O cenário não demonstra ganho realizado.

In [ ]:
# As duas alternativas atendem à mesma procura no cenário.
cols = ["Receita adicional (R$)", "Lucro bruto adicional (R$)", "Volume embalado a menos (L)"]
economia = cenario[cols].copy()
economia.loc["Total"] = economia.sum()
display(economia.round(2))
print(f"Diferença de desembolso em compras: R$ {cenario['Compras adicionais (R$)'].sum():.2f}")


**Q3.** Expliquem as duas oportunidades reveladas pelo cenário: vendas adicionais com lucro bruto positivo e menor necessidade de manter mercadorias em estoque. Usem os valores calculados e indiquem por que esses resultados podem orientar o abastecimento futuro.

**Resposta:**



## 5. Consolidem a interpretação — 7 min

Preencham o quadro com frases curtas e números pertinentes. Diferenciem evidências observadas e resultados hipotéticos. Não é necessário escolher uma prioridade definitiva nesta aula.

**Q4.** Registrem uma interpretação para cada dimensão investigada: espera e atendimento, horas de equipe, disponibilidade e procura não atendida, distribuição de estoques. Indiquem a evidência, sua relevância para a direção e uma condição ou informação ainda necessária.

| Dimensão | Evidência e significado | Relevância para a direção | Condição ou informação necessária |
|---|---|---|---|
| Espera e atendimento | | | |
| Horas de equipe | | | |
| Disponibilidade e procura não atendida | | | |
| Distribuição de estoques | | | |

## Critérios para avaliação da atividade

- Explica relações entre indicadores com evidências dos dois recortes.
- Compara alternativas de abastecimento com os valores calculados.
- Distingue oportunidade comercial e eficiência de estoque, sem confundir desembolso com lucro.
- Registra interpretações sustentadas e condições para a decisão estratégica.

Confiram as respostas com o professor e salvem o notebook preenchido. Sem envio ao AVA. A próxima aula não depende deste arquivo.

## Expectativa de respostas

**Q1.** No período completo, espera e aproveitamento apresentam r = −0,649, mas espera e resultado r = −0,233. Horas de equipe e receita: +0,337; horas de equipe e resultado: −0,249. Na faixa de movimento, espera e resultado: −0,099; equipe e resultado: −0,159. Receita e resultado após despesas são indicadores distintos; não se pode atribuir efeito causal. Disponibilidade e resultado: +0,393 e +0,325 nos dois recortes; procura não atendida e resultado: −0,046 e +0,017. L3 tem disponibilidade geral 96,80%, procura não atendida 9,09% e resultado médio −R$ 50,02; L4, 95,56%, 2,83% e R$ 317,65. L5 tem 2,77% e −R$ 136,15; L8, 5,67% e R$ 200,75. As descobertas locais são relevantes, mas não explicam todo o desempenho da rede.

**Q2.** No cenário de 24 a 29/11/2025, 13 cadernos (L3) e 19 resmas (L5), total de 32 unidades. Comprar: R$ 564,50 em compras adicionais e 65 unidades no estoque final das quatro combinações loja-produto. Aproveitar saldos existentes: R$ 0,00 em compras adicionais e 33 unidades no estoque final. Diferença de R$ 564,50 em desembolso e 32 unidades. Mantidas as reposições e vendas originais e a disponibilidade nos momentos necessários, os saldos diários calculados nas lojas de origem não chegam a zero. Não é uma ordem de transferência atual.

**Q3.** As 32 vendas adicionais hipotéticas gerariam R$ 837,29 de receita e R$ 272,79 de lucro bruto (R$ 121,03 em cadernos e R$ 151,76 em resmas), antes de eventuais despesas incrementais. A alternativa de aproveitar o estoque existente demandaria 32 unidades a menos no saldo final, correspondentes a 69,25 L de volume embalado e R$ 564,50 a menos em compras adicionais. Melhor planejar compras e reposições segundo a procura de cada loja pode favorecer vendas e reduzir estoques desnecessários e seus custos de manutenção. Não somar desembolso evitado a lucro bruto; volume embalado não comprova redução de aluguel. O cenário é hipotético, não ganho ocorrido.

**Q4.** Espera: associação negativa com aproveitamento; examinar atendimento sem inferir causalidade. Equipe: receita e resultado apresentam associações de sinais distintos; investigar custos e necessidade de pessoal. Disponibilidade: contraste L3/L4 e procura não atendida localizada; investigar padrões por produto, sem generalizar. Estoques: 23 dias de coincidência L3/L4 cadernos e 19 L5/L2 resmas; o cenário de uma semana indica que melhor distribuição futura pode atender procura com menores compras e estoques. Confirmar padrões de procura, reposições e nível de disponibilidade necessário. Aceitar outras interpretações fundamentadas. A prioridade será avaliada na A4.